# Green and Yellow Apple Detection, Depth, Tracking, and Counting

This notebook implements the requested orchard-video workflow: green/yellow apple labels, label-safe training augmentation, AAM–EBCA–FM–KWConv feature modules, AdaBins relative-depth attachment, and two-stage depth-aware ByteTrack-style tracking.

## Output contract

- `Dataset/frames/raw/`: extracted frames
- `Dataset/labels/seed_candidates/`: automatic HSV candidates — **review only**, not ground truth
- `Dataset/labels/verified/`: manually verified green/yellow YOLO labels
- `Dataset/augmentations/`: training-only augmented images and transformed labels
- `Dataset/yolo/`: temporally disjoint train/validation/test YOLO dataset
- `Results/`: architecture test, training, validation, tracked MP4, track CSV, and summary JSON

The raw video is not edited or overwritten.

In [ ]:
# 1) Setup — works whether Jupyter starts in the repository root or Codeing/
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
ROOT = next(
    candidate
    for candidate in (cwd, *cwd.parents)
    if (candidate / 'Dataset').exists() and (candidate / 'Codeing' / 'fruit_pipeline.py').exists()
)
CODE_DIR = ROOT / 'Codeing'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import fruit_pipeline as fp

paths = fp.project_paths(ROOT)
print('Project root:', paths.root)
print('Classes:', fp.CLASS_NAMES)

In [ ]:
# 2) Mandatory video preflight
# Do not continue to extraction until `playable` is True.
preflight = fp.video_preflight(paths.raw_video)
preflight

### Current input status

The supplied `DJI_0418.MP4` currently has no readable MP4 index (`moov` atom), so OpenCV and FFmpeg cannot extract even one frame. Re-export it from the original camera/device as a playable H.264 MP4, replace only `Dataset/DJI_0418.MP4`, re-run the preflight cell, then run the workflow below.

In [ ]:
# 3) Extract up to 800 temporally uniform frames for annotation.
# 5 fps is manageable for manual review and avoids near-identical frame leakage.
if preflight['playable']:
    frame_manifest = fp.extract_uniform_frames(paths, sample_fps=5.0, max_frames=800)
    print(f'Extracted {len(frame_manifest)} frames to {paths.frames}')
else:
    print('Waiting for a playable replacement video; no frames were written.')

In [ ]:
# 4) Generate GREEN/YELLOW HSV seed candidates and preview images.
# Green leaves can look like green apples, therefore these are never training labels by themselves.
seed_summary = fp.create_colour_seed_labels(paths)
seed_summary

## Human label verification (required)

Inspect the preview images in `Dataset/labels/seed_previews/`. For every frame, draw all true apple boxes and assign `g` (green) or `y` (yellow) with the next cell. It saves final labels in `Dataset/labels/verified/`. Draw no boxes and save an empty label file for a reviewed negative frame. This review step prevents foliage from becoming a false green-apple label.

In [ ]:
# 5) Run this one frame at a time on the local desktop.
# Draw all boxes, press Enter, then answer g/y/s for each ROI.
# The saved file is Dataset/labels/verified/<frame>.txt.
frame_to_review = next(iter(sorted(paths.frames.glob('*.jpg'))), None)
if frame_to_review is None:
    print('No extracted frames yet.')
else:
    print('Ready to review:', frame_to_review.name)
    # verified_boxes = fp.review_frame_with_opencv(paths, frame_to_review)

In [ ]:
# 6) After EVERY selected frame has a manually verified label file, make temporal splits.
# Splitting before augmentation prevents adjacent video frames and augmented copies from leaking into val/test.
# splits = fp.build_temporal_yolo_dataset(paths, train_fraction=0.70, val_fraction=0.10)
# print(splits)

In [ ]:
# 7) Augment TRAINING images only. Labels are transformed and saved alongside images.
# Covers partial leaf occlusion, dark/bright light, motion blur, and horizontal viewpoint reversal.
# Hue shifting is intentionally disabled because green and yellow are class labels.
# augmentation_summary = fp.augment_training_split(paths, seed=42)
# validation_report = fp.validate_yolo_dataset(paths)
# print(augmentation_summary)
# print(validation_report)

## Proposed AAM–EBCA–FM–KWConv feature modules

`fruit_pipeline.py` contains a testable proposed neck: AAM builds colour/shape/scale attention; EBCA exchanges P3/P4 information for overlap; FM aggregates local context for partial occlusion; and KWConv adaptively mixes kernel sizes for small and large apples. The smoke test checks P3/P4/P5 tensor interfaces and writes `Results/architecture_smoke_test.json`.

For a full Ultralytics model YAML integration, use a local Ultralytics source fork and register these modules in its parser as documented by Ultralytics; do not assume that defining a class in a notebook automatically changes a pretrained YOLO model.

In [ ]:
# 8) Optional architecture interface check (requires PyTorch).
# Install once in the active kernel: pip install torch
# fp.module_smoke_test(paths)

## Train and validate the reproducible detector baseline

Install the training dependencies in the notebook kernel first: `pip install -r Codeing/requirements.txt`. The baseline uses two detector classes (`green_apple`, `yellow_apple`) and saves weights, plots, and metrics under `Results/training/`. Run the AAM–EBCA–FM–KWConv ablation only after registering the custom modules in a local Ultralytics fork.

In [ ]:
# 9) Baseline training and validation.
RUN_TRAINING = False
if RUN_TRAINING:
    train_result = fp.train_baseline_yolov8(paths, model_name='yolov8n.pt', epochs=100, image_size=1024)
    best_weights = paths.training_results / 'yolov8_green_yellow_apple' / 'weights' / 'best.pt'
    validation_result = fp.validate_detector(paths, best_weights, image_size=1024)

## Relative AdaBins depth + depth-aware ByteTrack-style tracking

The RGB video contains no depth sensor data. AdaBins adds a per-pixel monocular estimate; in this orchard domain it must be treated as **relative depth** unless calibrated with orchard measurements. The tracker uses depth only as an association/gating signal, never as a claimed metric distance. It uses high-confidence matching first, then lower-confidence matching to recover temporarily missed apples, and a once-only line counter to prevent duplicate counts.

In [ ]:
# 10) Configure AdaBins only after cloning its official repository and installing its dependencies.
USE_ADABINS = False
ADABINS_REPOSITORY = ROOT / 'third_party' / 'AdaBins'
# The installed KITTI checkpoint is the outdoor option; use its output as relative depth only.
depth_estimator = fp.AdaBinsDepth(ADABINS_REPOSITORY, dataset='kitti') if USE_ADABINS else None

In [ ]:
# 11) Final inference: annotated video, CSV track log, and summary go to Results/inference/.
RUN_INFERENCE = False
BEST_WEIGHTS = paths.training_results / 'yolov8_green_yellow_apple' / 'weights' / 'best.pt'
if RUN_INFERENCE:
    frame_width, frame_height = preflight['width'], preflight['height']
    # Adjust this virtual line after inspecting the orchard camera direction.
    counting_line = ((frame_width // 2, 0), (frame_width // 2, frame_height - 1))
    summary = fp.run_depth_aware_tracking(
        paths,
        weights=BEST_WEIGHTS,
        adabins=depth_estimator,
        low_confidence=0.10,
        image_size=1024,
        counting_line=counting_line,
        count_direction='positive',
    )
    print(summary)

## Required result review

Before reporting research claims, inspect `Results/inference/tracked_apples.mp4` for missed apples, ID switches, duplicate counts, and foreground/background confusion. Report mAP50–95 per colour and per occlusion/size slice, then tracking IDF1/HOTA and count MAE against a manually verified temporal test sequence.